## Handoff Multi-Agent Design Pattern with LangChain

![handoff_workflow](./Images/handoff_workflow.png)

### Installing Utilities and Libraries

In [3]:
%pip install langchain-anthropic==1.5.4 anthropic==0.120.2 python-dotenv==1.2.2

Looking in indexes: https://pypi.org/simple/

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Setting up the Environment

In [6]:
import os
from dotenv import load_dotenv
from typing import Literal
from langchain.agents import AgentState, create_agent
from langchain.messages import AIMessage, ToolMessage
from langchain.tools import tool, ToolRuntime
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing_extensions import NotRequired

load_dotenv()
anthropic_api_key = os.getenv("CLAUDE_API_KEY")
anthropic_model_name = os.getenv("CLAUDE_MODEL_NAME")

### Instantiate the ChatAnthropic Class

In [7]:
from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(
    model_name = anthropic_model_name,
    api_key = anthropic_api_key
)

### Create the Agent State Class

In [8]:
# Define state with active_agent tracker
class MultiAgentState(AgentState):
    active_agent: NotRequired[str]

### Create the Sales Agent Transfer Tool

In [9]:
@tool
def transfer_to_sales(
    runtime: ToolRuntime,
) -> Command:
    """Transfer to the sales agent"""
    last_ai_message = next(
        msg for msg in reversed(runtime.state["messages"]) if isinstance(msg, AIMessage)
    )

    transfer_message = ToolMessage(
        content = "Transferred to sales agent from support agent",
        tool_call_id = runtime.tool_call_id
    )

    return Command(
        goto="sales_agent",
        update = {
            "active_agent": "sales_agent",
            "messages": [last_ai_message, transfer_message]
        },
        graph = Command.PARENT
    )

### Create the Support Agent Transfer Tool

In [10]:
@tool
def transfer_to_support(
    runtime: ToolRuntime,
) -> Command:
    """Transfer to the support agent."""
    last_ai_message = next(
        msg for msg in reversed(runtime.state["messages"]) if isinstance(msg, AIMessage)
    )
    transfer_message = ToolMessage(
        content="Transferred to support agent from sales agent",
        tool_call_id=runtime.tool_call_id,
    )
    return Command(
        goto="support_agent",
        update={
            "active_agent": "support_agent",
            "messages": [last_ai_message, transfer_message],
        },
        graph=Command.PARENT,
    )


### Create Agents with the Handoff Tools

In [11]:
# Create the sales agent
sales_agent = create_agent(
    model = model,
    tools = [transfer_to_support],
    system_prompt = "You are a sales agent. Help with sales inquiries. If asked about technical issues or support, transfer to the support agent."
)

# Create the support agent
support_agent = create_agent(
    model=model,
    tools=[transfer_to_sales],
    system_prompt="You are a support agent. Help with technical issues. If asked about pricing or purchasing, transfer to the sales agent.",
)

### Create Agent Nodes that Invoke the Agents

In [13]:
# Create the sales agent node
def call_sales_agent(state: MultiAgentState) -> Command:
    """Node that calls the sales agent."""
    response = sales_agent.invoke(state)
    return response

# Create the support agent node
def call_support_agent(state: MultiAgentState) -> Command:
    """Node that calls the support agent."""
    response = support_agent.invoke(state)
    return response

### Create the Workflow Routers

In [14]:
# Create router that checks if we should end or continue
def route_after_agent(
    state: MultiAgentState,
) -> Literal["sales_agent", "support_agent", "__end__"]:
    """Route based on active_agent, or END if the agent finished without handoff."""
    messages = state.get("messages", [])

    # Check the last message - if it's an AIMessage without tool calls, we're done
    if messages:
        last_msg = messages[-1]
        if isinstance(last_msg, AIMessage) and not last_msg.tool_calls:
            return "__end__"

    # Otherwise route to the active agent
    active = state.get("active_agent", "sales_agent")
    return active if active else "sales_agent"

# Create the Initial Router
def route_initial(
    state: MultiAgentState,
) -> Literal["sales_agent", "support_agent"]:
    """Route to the active agent based on state, default to sales agent."""
    return state.get("active_agent") or "sales_agent"

### Build the Workflow Graph

In [15]:
# Build the graph
builder = StateGraph(MultiAgentState)
builder.add_node("sales_agent", call_sales_agent)
builder.add_node("support_agent", call_support_agent)

# Start with conditional routing based on initial active_agent
builder.add_conditional_edges(START, route_initial, ["sales_agent", "support_agent"])

# After each agent, check if we should end or route to another agent
builder.add_conditional_edges(
    "sales_agent", route_after_agent, ["sales_agent", "support_agent", END]
)
builder.add_conditional_edges(
    "support_agent", route_after_agent, ["sales_agent", "support_agent", END]
)

graph = builder.compile()

### Generate the Mermaid Diagram of the Workflow

In [ ]:
png = graph.get_graph().draw_mermaid_png()

with open("handoff_workflow.png", "wb") as f:
    f.write(png)

### Invoke the Workflow

You can begin by running this sample query - `I am having issues with my account login. Can you please help ?`

In [17]:
# Run the multi-agent system as an interactive chat

state = {
    "messages": [],
    "active_agent": "sales_agent",
}

print("Multi-Agent Chat")
print("Type 'exit' to end the conversation.\n")


while True:

    # Get user input
    user_input = input("You: ")

    # Exit the chat
    if user_input.lower() in ["exit", "quit"]:
        print("Chat ended.")
        break

    # Add the new user message to the existing conversation
    state["messages"].append(
        {
            "role": "user",
            "content": user_input,
        }
    )

    # Invoke the graph
    result = graph.invoke(state)

    # Find the latest AI response
    last_ai_message = next(
        msg
        for msg in reversed(result["messages"])
        if isinstance(msg, AIMessage)
    )

    for msg in result["messages"]:
     msg.pretty_print()

    # Preserve the graph state for the next turn
    state = result

Multi-Agent Chat
Type 'exit' to end the conversation.

================================ Human Message =================================

i need refund
================================== Ai Message ==================================

[{'signature': 'Er0CCpABCBIYAipAD+/e+RPwJU/TTUtD8hYXIpfSaPWc19MkypQNIbR4pI88EOqk1xHyIomnTMK52+76mozD5y8bRHSh9XEAJUDwsDIPY2xhdWRlLXNvbm5ldC01OABCCHRoaW5raW5nWiQ4YTdhNWVmNC1iYjMxLTQwYTItYTA0My02NjVhYjJkZGU4NDaoAd/L3tUGEgzXh7FpmuEg5Tiv/v8aDBwBnsmsQqAMwot71SIw+JiVZjmq2GeMpmwE2XxTvtLfwtI1XEy4K0NAGL1b3QclVBWUdW6FxHNrYMgTW8aSKlqfz8Rwv3TEWyXpw+fbbZE1W+/yO6uoTa0QsRykHywje0Od3zYTx/PUZFJOBFpX72Pp8GfvQBqsUGD00z4wUBq8hsrxUKzcljmg6qJCTYv5EH4WPbHeUxP1KAMYAQ==', 'thinking': '', 'type': 'thinking'}, {'text': "I can help direct you to the right place for that. Refunds are handled by our support team, so I'll transfer you now.", 'type': 'text'}, {'id': 'toolu_01KzzH4uJLk6pKkvGcGBC1Jo', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'transfer_to_support', 'type': 'tool_use